In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import warnings
import matplotlib.pyplot as plt
import time
from functools import partial  # <-- IMPORT PENTING UNTUK MENCEGAH PICKLING ERROR

warnings.filterwarnings("ignore")

# ==========================================
# 1. LOAD DATA DAN SCALER
# ==========================================
base_path = './split/'

train = pd.read_csv(base_path + '90training_normalized.csv')
test  = pd.read_csv(base_path + '10testing_normalized.csv')
test_asli = pd.read_csv(base_path + '9testing.csv') 

X_train = train.drop(columns=['Produksi'])
y_train = train['Produksi']

X_test = test.drop(columns=['Produksi'])
y_test = test['Produksi']

scaler_y = joblib.load(base_path + '90training_scaler_y.save')

# Preprocessing data test_asli untuk output CSV
bulan_map = {
    'Januari':1, 'Februari':2, 'Maret':3, 'April':4,
    'Mei':5, 'Juni':6, 'Juli':7, 'Agustus':8,
    'September':9, 'Oktober':10, 'November':11, 'Desember':12
}
test_asli[['Nama_Bulan', 'Tahun']] = test_asli['Periode'].str.split(' ', expand=True)
test_asli['Bulan'] = test_asli['Nama_Bulan'].map(bulan_map)
test_asli['Tahun'] = test_asli['Tahun'].astype(int)
test_asli.drop(columns=['Periode', 'Nama_Bulan'], inplace=True)

print(f"Data shape - X_train: {X_train.shape}, X_test: {X_test.shape}")

# ==========================================
# 2. FUNGSI KERNEL & EVALUASI SVR (TARGET RMSE)
# ==========================================
def hitung_anova_rbf(X, Y, gamma, degree=2):
    X = np.asarray(X)
    Y = np.asarray(Y)
    K = np.zeros((X.shape[0], Y.shape[0]))
    
    for k in range(X.shape[1]):
        diff_sq = (X[:, k].reshape(-1, 1) - Y[:, k].reshape(1, -1)) ** 2
        K += np.exp(-gamma * diff_sq)
        
    K = K / X.shape[1] 
    
    return K ** degree

def evaluate_svr(params):
    C, epsilon, gamma = params
    try:
        # Gunakan partial untuk menitipkan gamma ke fungsi kernel
        kernel_func = partial(hitung_anova_rbf, gamma=gamma)
        
        # max_iter=10000 mencegah SVR nyangkut pada parameter yang jelek
        model = SVR(kernel=kernel_func, C=C, epsilon=epsilon, max_iter=10000)
        model.fit(X_train, y_train)
        
        y_pred_norm = model.predict(X_test)
        
        # Denormalisasi agar RMSE dihitung dalam satuan asli (Ton)
        y_pred_asli = scaler_y.inverse_transform(y_pred_norm.reshape(-1, 1)).flatten()
        y_test_asli = scaler_y.inverse_transform(y_test.values.reshape(-1, 1)).flatten()
        
        # Mencegah nilai negatif
        y_pred_asli = np.clip(y_pred_asli, 0, None)
        
        # Hitung RMSE
        rmse_score = np.sqrt(mean_squared_error(y_test_asli, y_pred_asli))
        
        return rmse_score, y_pred_norm 
    except Exception as e:
        return float('inf'), None

# ==========================================
# 3. FUNGSI EXPORT HASIL CSV
# ==========================================
def export_hasil_terbaik(n_particles, global_best, global_best_pred):
    if global_best_pred is None:
        return

    try:
        y_pred_asli = scaler_y.inverse_transform(global_best_pred.reshape(-1,1)).flatten()
        y_pred_asli = np.clip(y_pred_asli, 0, None)
        y_test_asli = scaler_y.inverse_transform(y_test.values.reshape(-1,1)).flatten()

        hasil = test_asli.copy()
        hasil['Produksi_Asli'] = y_test_asli.round(2)
        hasil['Prediksi_Ton']  = y_pred_asli.round(2)
        hasil = hasil[['Tahun', 'Bulan', 'Kabupaten/Kota', 'Produksi_Asli', 'Prediksi_Ton']]
        
        # Tambahkan kata rmse pada nama file
        nama_file_csv = f'hasil_terbaik_anova_rmse_{n_particles}_partikel.csv'
        hasil.to_csv(nama_file_csv, index=False)
    except Exception as e:
        print(f"  [!] Gagal ekspor CSV: {e}")

# ==========================================
# 4. TEST KERNEL SEBELUM PSO DIMULAI
# ==========================================
def test_kernel():
    print("\n>>> Testing kernel ANOVA...")
    try:
        gamma_test, C_test, epsilon_test = 0.01, 50, 0.01
        
        # Test dengan partial
        kernel_test = partial(hitung_anova_rbf, gamma=gamma_test)
        model_test = SVR(kernel=kernel_test, C=C_test, epsilon=epsilon_test, max_iter=1000)
        
        model_test.fit(X_train[:100], y_train[:100])
        y_pred_test_norm = model_test.predict(X_test[:20])
        
        if np.any(np.isnan(y_pred_test_norm)) or np.any(np.isinf(y_pred_test_norm)):
            print("[!] Kernel test GAGAL: prediksi mengandung NaN atau Inf")
            return False
            
        # Denormalisasi untuk test RMSE
        y_pred_test_asli = scaler_y.inverse_transform(y_pred_test_norm.reshape(-1, 1)).flatten()
        y_test_asli_subset = scaler_y.inverse_transform(y_test[:20].values.reshape(-1, 1)).flatten()
        
        rmse_test = np.sqrt(mean_squared_error(y_test_asli_subset, y_pred_test_asli))
        print(f"✓ Kernel test OK - RMSE: {rmse_test:.4f}\n")
        return True
        
    except Exception as e:
        print(f"[!] Kernel test GAGAL: {e}\n")
        return False

# ==========================================
# 5. ALGORITMA PSO UTAMA DENGAN RESUME
# ==========================================
def pso_auto_resume(n_particles, target_iter, timeout=10):
    np.random.seed(42)
    # Nama checkpoint diubah agar tidak menimpa versi MAPE
    nama_checkpoint = f'pso_anova_rmse_{n_particles}_partikel.save'
    
    lb = np.array([1, 0.000001, 0.00001])
    ub = np.array([1000, 0.1, 100])
    w, c1, c2 = 0.7, 1.5, 1.5

    if os.path.exists(nama_checkpoint):
        print(f">>> File checkpoint ditemukan! Memuat data {n_particles} partikel...")
        state = joblib.load(nama_checkpoint)
        
        start_iter = state['iterasi_terakhir']
        particles = state['particles']
        velocities = state['velocities']
        personal_best = state['personal_best']
        personal_best_score = state['personal_best_score']
        global_best = state['global_best']
        global_best_score = state['global_best_score']
        global_best_pred = state['global_best_pred']
        rmse_history = state['rmse_history']
        print(f">>> Melanjutkan dari iterasi ke-{start_iter + 1} menuju {target_iter}...\n")
    else:
        print(f">>> Memulai PSO dari awal untuk {n_particles} partikel (Target: RMSE)...\n")
        start_iter = 0
        particles = np.random.uniform(lb, ub, (n_particles, 3))
        velocities = np.zeros((n_particles, 3))
        personal_best = particles.copy()
        personal_best_score = np.array([float('inf')] * n_particles)
        global_best = None
        global_best_score = float('inf')
        global_best_pred = None
        rmse_history = []

    if start_iter >= target_iter:
        print("Target iterasi sudah tercapai di run sebelumnya.")
        return rmse_history

    failed_count = 0
    for i in range(start_iter, target_iter):
        print(f"iterasi ke {i+1}/{target_iter} :")
        iter_success = False
        
        for j in range(n_particles):
            start = time.time()
            score, pred = evaluate_svr(particles[j])
            elapsed = time.time() - start
            
            if elapsed > timeout:
                score, pred = float('inf'), None

            if score < personal_best_score[j]:
                personal_best[j] = particles[j]
                personal_best_score[j] = score

                if score < global_best_score:
                    global_best = particles[j]
                    global_best_score = score
                    global_best_pred = pred
                    iter_success = True
            
            score_tampil = f"{score:.4f}" if score != float('inf') else "inf"
            print(f"partikel {j+1}/{n_particles}, RMSE : {score_tampil}, waktu: {elapsed:.4f} s.")

        if not iter_success and global_best_score == float('inf'):
            failed_count += 1
        else:
            failed_count = 0
            
        rmse_history.append(global_best_score)

        for j in range(n_particles):
            r1, r2 = np.random.rand(), np.random.rand()
            gb = global_best if global_best is not None else personal_best[j]
            velocities[j] = (
                w * velocities[j]
                + c1 * r1 * (personal_best[j] - particles[j])
                + c2 * r2 * (gb - particles[j])
            )
            particles[j] += velocities[j]
            particles[j] = np.clip(particles[j], lb, ub)

        print(f"--- Iterasi {i+1} Selesai | Global Best RMSE: {global_best_score:.4f} ---\n")

        state = {
            'iterasi_terakhir': i + 1,
            'particles': particles,
            'velocities': velocities,
            'personal_best': personal_best,
            'personal_best_score': personal_best_score,
            'global_best': global_best,
            'global_best_score': global_best_score,
            'global_best_pred': global_best_pred,
            'rmse_history': rmse_history
        }
        joblib.dump(state, nama_checkpoint)
        export_hasil_terbaik(n_particles, global_best, global_best_pred)
        
        if failed_count >= 5:
            print("\n[!] WARNING: 5 iterasi berturut-turut gagal! Cek rentang parameter.")

    return rmse_history

# ==========================================
# 6. RUN PROGRAM 
# ==========================================
JUMLAH_PARTIKEL = 100
TARGET_ITERASI = 150

if test_kernel():
    history = pso_auto_resume(n_particles=JUMLAH_PARTIKEL, target_iter=TARGET_ITERASI)

    # SIMPAN MODEL TERBAIK DAN EVALUASI FINAL
    state = joblib.load(f'pso_anova_rmse_{JUMLAH_PARTIKEL}_partikel.save')
    best_params = state['global_best']

    if best_params is not None:
        C_best, epsilon_best, gamma_best = best_params
        print('\n' + '='*50)
        print('HASIL AKHIR PARAMETER TERBAIK (Target RMSE):')
        print('='*50)
        print('C = {:.4f}'.format(C_best))
        print('e = {:.6f}'.format(epsilon_best))
        print('gamma = {:.4f}'.format(gamma_best))

        # Train model final dengan parameter terbaik
        kernel_func_best = partial(hitung_anova_rbf, gamma=gamma_best)
        model_best = SVR(kernel=kernel_func_best, C=C_best, epsilon=epsilon_best)
        model_best.fit(X_train, y_train)
        
        joblib.dump(model_best, f'model_svr_anova_rmse_p{JUMLAH_PARTIKEL}i{TARGET_ITERASI}.save')
        print(f'\n✓ Model final tersimpan di: model_svr_anova_rmse_p{JUMLAH_PARTIKEL}i{TARGET_ITERASI}.save')

        # Prediksi Final
        y_train_pred = model_best.predict(X_train)
        y_test_pred = model_best.predict(X_test)

        # Denormalisasi Final
        y_train_pred_asli = scaler_y.inverse_transform(y_train_pred.reshape(-1,1)).flatten()
        y_train_asli = scaler_y.inverse_transform(y_train.values.reshape(-1,1)).flatten()
        y_test_pred_asli = scaler_y.inverse_transform(y_test_pred.reshape(-1,1)).flatten()
        y_test_asli = scaler_y.inverse_transform(y_test.values.reshape(-1,1)).flatten()

        # Mencegah nilai negatif
        y_train_pred_asli = np.clip(y_train_pred_asli, 0, None)
        y_test_pred_asli = np.clip(y_test_pred_asli, 0, None)

        print('\n' + '='*50)
        print('EVALUASI TRAINING :')
        print('='*50)
        print('RMSE: {:.4f}'.format(np.sqrt(mean_squared_error(y_train_asli, y_train_pred_asli))))
        print('R2:   {:.4f}'.format(r2_score(y_train_asli, y_train_pred_asli)))

        print('\n' + '='*50)
        print('EVALUASI TESTING :')
        print('='*50)
        print('RMSE: {:.4f}'.format(np.sqrt(mean_squared_error(y_test_asli, y_test_pred_asli))))
        print('R2:   {:.4f}'.format(r2_score(y_test_asli, y_test_pred_asli)))

        if len(history) > 0:
            plt.figure(figsize=(10, 6))
            plt.plot(history, linewidth=2)
            plt.title(f"Konvergensi PSO ANOVA ({JUMLAH_PARTIKEL} Partikel, {TARGET_ITERASI} Iterasi)", fontsize=14)
            plt.xlabel("Iterasi", fontsize=12)
            plt.ylabel("RMSE (Ton)", fontsize=12)
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()

        print("\n" + "="*50)
        print(">>> SELURUH PROSES SELESAI <<<")
        print("="*50)
    else:
        print("\n[!] Program dihentikan karena tidak ada parameter yang berhasil dikalkulasi.")
else:
    print("\n[!] Kernel Test gagal, program PSO tidak dijalankan.")